In [ ]:
ml_run_id = 'PASTE_TRAIN_ML_RUN_GUID'
scoring_run_id = 'PASTE_NEW_SCORING_RUN_GUID'


In [ ]:
import json, uuid, hashlib
import datetime as dt
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import notebookutils
from sklearn.ensemble import HistGradientBoostingRegressor
from mlflow.models import infer_signature
from pyspark.sql import functions as F
FEATURES = ['qty_lag_1', 'qty_sum_7d', 'qty_sum_28d', 'qty_avg_28d', 'qty_std_28d',
    'zero_days_28d', 'weekday', 'month']
FEATURE_VERSION = 'sales_qty_v1'
def require(ok, msg):
    if not bool(ok):
        raise ValueError(msg)
def features(obs):
    out = []
    for key, g in obs.groupby('product_key', sort=False):
        g = g.sort_values('observation_date').copy()
        require(not g.observation_date.duplicated().any(), 'Duplicate product-day')
        require(g.observation_date.diff().dropna().eq(pd.Timedelta(days=1)).all(),
            'Sparse calendar: ROW windows invalid')
        q = g.sold_qty.astype(float)
        prev = q.shift(1)
        g['qty_lag_1'] = prev
        g['qty_sum_7d'] = prev.rolling(7, min_periods=7).sum()
        g['qty_sum_28d'] = prev.rolling(28, min_periods=28).sum()
        g['qty_avg_28d'] = prev.rolling(28, min_periods=28).mean()
        g['qty_std_28d'] = prev.rolling(28, min_periods=28).std(ddof=0)
        g['zero_days_28d'] = prev.eq(0).astype(float).where(prev.notna()).rolling(28,
            min_periods=28).sum()
        g['weekday'] = g.observation_date.dt.dayofweek
        g['month'] = g.observation_date.dt.month
        g['label_qty_next_7d'] = q.iloc[::-1].rolling(7, min_periods=7).sum().iloc[::-1]
        g['label_end'] = g.observation_date + pd.Timedelta(days=6)
        g['baseline_qty'] = g.qty_avg_28d * 7
        out.append(g)
    return pd.concat(out, ignore_index=True)
def score(obs, elig, model, cutoff, scoring_run_id, version, dsid):
    cutoff = pd.Timestamp(cutoff).normalize()
    elig = elig.copy()
    elig['active_start'] = pd.to_datetime(elig.active_start)
    elig['active_end'] = pd.to_datetime(elig.active_end)
    e = elig[(elig.active_start <= cutoff) & (elig.active_end.isna() | (elig.active_end >= cutoff)) & elig.finished_goods_flag.astype(bool)]
    require(obs.observation_date.max() < cutoff, 'Scoring input must stop before cutoff')
    future = pd.DataFrame({'product_key': e.product_key.astype(int),
        'observation_date': cutoff, 'sold_qty': np.nan, 'net_amount': np.nan,
        'availability_mode': 'RECONSTRUCTED',
        'source_gold_run_id': obs.source_gold_run_id.iloc[0]})
    joined = pd.concat([obs, future], ignore_index=True)
    f = features(joined)
    f = f[f.observation_date == cutoff].dropna(subset=FEATURES)
    require(len(f) > 0, 'No products with 28 complete history days')
    now = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
    rows = pd.DataFrame({'scoring_run_id': scoring_run_id,
        'product_key': f.product_key.to_numpy(), 'cutoff_date': cutoff.date(),
        'horizon_days': 7, 'predicted_qty': np.maximum(0, model.predict(f[FEATURES])),
        'baseline_qty': f.baseline_qty.to_numpy(), 'model_version': version,
        'dataset_id': dsid, 'feature_version': FEATURE_VERSION,
        'source_gold_run_id': obs.source_gold_run_id.iloc[0],
        'availability_mode': 'RECONSTRUCTED', 'scored_at': now,
        'valid_until': now + dt.timedelta(hours=24)})
    return rows


In [ ]:
assert len(ml_run_id) == 36 and len(scoring_run_id) == 36
artifact_root = f'Files/gold_ml_artifacts/{ml_run_id}'
meta = json.loads(notebookutils.fs.head(artifact_root + '/training_result.json', 65536))
obs = spark.read.parquet(artifact_root + '/observations').toPandas()
elig = spark.read.parquet(artifact_root + '/eligibility').toPandas()
obs['observation_date'] = pd.to_datetime(obs['observation_date'])
cutoff = pd.Timestamp(meta['coverage_end']) + pd.Timedelta(days=1)
model = mlflow.sklearn.load_model(f"models:/{meta['model_name']}/{meta['model_version']}")
pred = score(obs, elig, model, cutoff, scoring_run_id, meta['model_version'],
    meta['dataset_id'])
assert len(pred) > 0 and pred['predicted_qty'].ge(0).all()
assert not pred.duplicated(['product_key', 'cutoff_date', 'horizon_days']).any()
print('Scored products', len(pred), 'cutoff', cutoff.date())


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType
cols = [('scoring_run_id', StringType()), ('product_key', IntegerType()), ('cutoff_date',
    DateType()), ('horizon_days', IntegerType()), ('predicted_qty', DoubleType()),
    ('baseline_qty', DoubleType()), ('model_version', StringType()), ('dataset_id',
    StringType()), ('feature_version', StringType()), ('source_gold_run_id',
    StringType()), ('availability_mode', StringType()), ('scored_at', TimestampType()),
    ('valid_until', TimestampType())]
rows = []
for r in pred.to_dict('records'):
    rows.append((str(r['scoring_run_id']), int(r['product_key']),
        pd.Timestamp(r['cutoff_date']).date(), int(r['horizon_days']),
        float(r['predicted_qty']), float(r['baseline_qty']), str(r['model_version']),
        str(r['dataset_id']), str(r['feature_version']), str(r['source_gold_run_id']),
        str(r['availability_mode']), pd.Timestamp(r['scored_at']).to_pydatetime(),
        pd.Timestamp(r['valid_until']).to_pydatetime()))
pred_spark = spark.createDataFrame(rows, StructType([StructField(n, t, False) for n,
    t in cols]))
path = f'Files/gold_ml_scoring/{scoring_run_id}/predictions'
pred_spark.write.mode('errorifexists').parquet(path)
manifest = spark.createDataFrame([(scoring_run_id, int(len(pred)),
    str(meta['gold_run_id']))],
    'scoring_run_id string, expected_prediction_count int, source_gold_run_id string')
manifest.write.format('delta').mode('append').saveAsTable('dbo.gold_scoring_manifest')
print(path)
print('Expected count', len(pred))
